<style>
.jp-CodeCell .jp-InputArea, div.input_area {background:#eef8ee !important; border-left:5px solid #2e7d32 !important;}
.jp-CodeCell pre, div.input_area pre {color:#145a20 !important;}
.note-box {background:#eef5ff; border-left:5px solid #2f5597; padding:12px; margin:10px 0;}
.warning-box {background:#fff4e5; border-left:5px solid #d97706; padding:12px; margin:10px 0;}
.success-box {background:#edf7ed; border-left:5px solid #2e7d32; padding:12px; margin:10px 0;}
.decision-box {background:#f7f4ff; border-left:5px solid #6b46c1; padding:12px; margin:10px 0;}
</style>

# SBI 812: Machine Learning in Bioinformatics
## Lecture 2 — Exploration and Preprocessing

**Duration:** 3 hours  
**Level:** Introductory  
**Prerequisite:** Lecture 1 — machine-learning language and basic pandas inspection  
**Practical outcome:** Create a defensible exploratory and preprocessing report for a biological dataset

This lecture uses the supplied file:

```text
lecture02_bacterial_expression_preprocessing.csv
```

Place the CSV file in the same directory as this notebook before beginning.

## 1. Why this lecture matters

A machine-learning algorithm cannot repair a poorly understood dataset. Before modelling, investigate what the measurements look like, whether values are missing or implausible, whether categories are consistent, whether samples are duplicated, and whether features are measured on comparable scales.

Preprocessing is not simply a list of commands. Every change to biological data is a scientific decision.

For example, a missing gene-expression value could represent:

- a failed measurement;
- expression below the detection limit;
- a file-processing problem;
- an uncollected sample; or
- a genuine absence represented incorrectly.

Each explanation could require a different response. A reliable analysis therefore follows this habit:

> **Detect → investigate → decide → document → verify**

<div class="warning-box"><b>Never describe data as “clean” merely because a command ran without an error.</b></div>

## Learning outcomes

By the end of the lecture, you should be able to:

1. explain exploratory data analysis in plain language;
2. distinguish identifiers, metadata, numerical features and categorical features;
3. inspect distributions using summaries and visualizations;
4. identify missing values, duplicate identifiers, inconsistent categories and implausible measurements;
5. explain why outliers require investigation rather than automatic deletion;
6. convert a wrongly stored numerical column safely;
7. choose and document a simple missing-value strategy;
8. explain why categorical variables require encoding;
9. explain why some algorithms require scaling;
10. apply one-hot encoding and standard scaling for demonstration;
11. recognize that preprocessing choices can cause data leakage; and
12. produce a reproducible preprocessing decision log and save prepared data.

# Part 1 — The dataset and the biological question

The dataset is synthetic. Each row represents one bacterial culture sample. It contains measurements from three species, two treatments, three experimental batches and several time points.

The broad future question is:

> Can measured biological features help describe or eventually predict treatment response?

No model is trained in this lecture. The aim is to prepare data so that later modelling can be defensible.

## Data dictionary

| Column | Meaning | Expected type | Possible role |
|---|---|---|---|
| `sample_id` | sample identifier | text | identifier; not a biological feature |
| `species` | bacterial species | category | biological metadata/categorical feature |
| `treatment` | control or drug exposure | category | experimental metadata/categorical feature |
| `batch` | laboratory batch | category | technical metadata; possible confounder |
| `time_h` | collection time in hours | number | numerical metadata/possible feature |
| `gene_a` | normalized expression of gene A | number | numerical feature |
| `gene_b` | normalized expression of gene B | number | numerical feature |
| `gene_c` | normalized expression of gene C | number | numerical feature |
| `growth_rate` | growth per hour | number | numerical feature or future regression target |
| `response` | sensitive or resistant | category | possible classification target |

The role of a column depends on the scientific question. A column is not automatically a feature merely because it appears in the file.

# Part 2 — Import the tools

Three main libraries are used:

- **pandas** for tables;
- **Matplotlib** for figure control;
- **Seaborn** for statistical visualizations.

`StandardScaler` from scikit-learn is also used to demonstrate scaling. No model is fitted.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## How to read the import commands

```python
import pandas as pd
```

loads pandas and gives it the short name `pd`.

```python
from sklearn.preprocessing import StandardScaler
```

means: from the `preprocessing` part of scikit-learn, import the tool named `StandardScaler`.

The imports make tools available; they do not change the dataset.

`sns.set_theme(style="whitegrid")` asks Seaborn to use a white plotting background with grid lines. `pd.set_option("display.max_columns", None)` tells pandas not to hide columns when displaying a wide table; `None` means no fixed maximum. These two settings change presentation, not the stored measurements.

## How to read Python and pandas code

Before continuing, learn the small pieces from which pandas commands are built.

### 1. The equals sign stores a result

```python
filename = "data.csv"
```

The right-hand side is evaluated first. The text `"data.csv"` is then stored under the name `filename`. In Python, `=` means **assign or store**; it does not ask whether two things are equal.

### 2. Quotation marks identify text

`"gene_a"` is text. `gene_a` without quotation marks would be interpreted as the name of a Python variable. Column names are therefore usually written inside quotation marks.

### 3. Square brackets select or construct items

```python
raw_data["gene_a"]
```

selects one column and returns a pandas **Series**, which is a one-dimensional labelled object.

```python
raw_data[["sample_id", "gene_a"]]
```

uses an inner list of column names, so it selects several columns and returns a two-dimensional **DataFrame**.

```python
columns = ["gene_a", "gene_b", "gene_c"]
```

constructs a Python list. A list is an ordered collection of items.

### 4. A dot accesses something belonging to an object

In `raw_data.head()`, the dot means “use `head` that belongs to `raw_data`.” A DataFrame provides methods for inspecting and transforming its contents.

### 5. Parentheses call a function or method

`head()` tells Python to perform the `head` operation. Information placed inside the parentheses is passed to the operation. For example, `head(3)` requests three rows.

Some names after a dot are **attributes**, not methods. They provide stored information and therefore have no parentheses:

```python
raw_data.shape
raw_data.dtypes
```

### 6. Named arguments make an instruction specific

```python
raw_data["batch"].value_counts(dropna=False)
```

Here `dropna=False` is a named argument. It tells `value_counts()` not to hide missing values. `False` and `True` are Boolean values.

### 7. Comments explain code

Any text following `#` is a comment for the reader. Python does not execute it.

Do not try to memorize whole commands. Read each command as a combination of these smaller pieces.

# Part 3 — Load the supplied CSV

Always preserve the original imported DataFrame. In this notebook it is called `raw_data`. A copy is created before any values are changed.

In [ ]:
filename = "lecture02_bacterial_expression_preprocessing.csv"
raw_data = pd.read_csv(filename)
raw_data.head()

Read the command from right to left:

1. pandas reads the named CSV file;
2. it creates a DataFrame;
3. the DataFrame is stored as `raw_data`;
4. `.head()` displays the first five rows.

`pd` is the short name assigned to pandas during import. The dot in `pd.read_csv` means that `read_csv` belongs to pandas. `read_csv(filename)` calls that function and supplies the stored filename. The empty parentheses in `head()` use its default of five rows.

If `FileNotFoundError` appears, first check where the notebook is running and which CSV files are visible.

In [ ]:
from pathlib import Path

current_directory = Path.cwd()
csv_files = list(current_directory.glob("*.csv"))

print("Current directory:", current_directory)
print("CSV files found:", csv_files)

`Path` is a tool for working with file locations. `Path.cwd()` returns the **current working directory**—the folder in which Python is currently looking for relative filenames. `.glob("*.csv")` searches that folder: the asterisk `*` means “any filename,” while `.csv` restricts the result to CSV files. `list(...)` collects the search results into a list that can be displayed.

## First audit

Before plotting or changing values, answer:

- How many rows and columns were loaded?
- Do column names match the data dictionary?
- Are numerical columns actually numerical?
- Which columns contain missing values?
- Does each sample ID occur once?
- Are categorical spellings consistent?

In [ ]:
print("Shape:", raw_data.shape)
print("\nColumns:")
print(raw_data.columns.tolist())
print("\nData types:")
print(raw_data.dtypes)
print("\nMissing values:")
print(raw_data.isna().sum())

## Interpretation of the first audit

`shape` reports `(rows, columns)`. A row should represent one sample and a column one recorded field.

### Reading the audit syntax

- `raw_data.shape` is an attribute containing two numbers. It has no parentheses because it is information, not an action.
- `raw_data.columns` is the column-name index. `.tolist()` converts it to an ordinary Python list for clearer printing.
- `raw_data.dtypes` reports the stored data type of every column.
- `raw_data.isna()` creates a table of `True` and `False` values. `True` means the corresponding cell is missing.
- `.sum()` counts the `True` values in each column because pandas treats `True` as 1 and `False` as 0 in this calculation.
- `"\n"` inside printed text starts a new line. The backslash gives the letter `n` this special meaning.

Pay particular attention to `gene_a`. It should be numerical, but pandas reports it as `object`. This usually means at least one entry contains text or mixed types. Do not force conversion before locating the problematic entry.

In [ ]:
# Step 1: Try to convert gene_a into numbers.
gene_a_as_numbers = pd.to_numeric(raw_data["gene_a"], errors="coerce")

# Step 2: Ask which converted values are missing.
gene_a_is_missing = gene_a_as_numbers.isna()

# Step 3: Use the True/False result to select the affected rows.
rows_with_gene_a_problem = raw_data[gene_a_is_missing]

# Step 4: Display only the two columns that help us investigate.
rows_with_gene_a_problem[["sample_id", "gene_a"]]

The operation is divided into four steps. Each new variable stores an intermediate result that can be examined before continuing.

`pd.to_numeric(..., errors="coerce")` attempts numerical conversion. Values that cannot be converted become `NaN` temporarily. This temporary result is used only to locate suspicious entries.

`NaN` means a missing or unavailable numerical value. It does not explain why the value is missing.

### Reading the conversion code precisely

- `pd.to_numeric(...)` asks pandas to convert values to numbers.
- The first argument, `raw_data["gene_a"]`, is the Series to convert.
- The named argument `errors="coerce"` says: replace an unconvertible entry with `NaN` instead of stopping with an error. **Coerce** here means force invalid text into the missing-value representation.
- `.isna()` returns one Boolean value per row.
- `raw_data[gene_a_is_missing]` is **Boolean filtering**. pandas keeps rows where the matching Boolean value is `True`.
- `["sample_id", "gene_a"]` is a list of two column names. Using that list inside another pair of square brackets selects both columns.

## Guided check 1

Write code to display:

1. the final three rows;
2. all unique treatment labels;
3. all unique species labels;
4. counts for each batch; and
5. every row containing at least one missing value.

In [ ]:
# Write your code here before opening the solution.

### Guided check 1 — solution

In [ ]:
print(raw_data.tail(3))
print(raw_data["treatment"].unique())
print(raw_data["species"].unique())
print(raw_data["batch"].value_counts())
missing_row_mask = raw_data.isna().any(axis=1)
print(raw_data[missing_row_mask])

### Reading the guided-check solution

- `.tail(3)` returns the final three rows.
- `.unique()` returns each distinct value once; it does not count them.
- `.value_counts()` returns each distinct value together with its frequency.
- `.isna()` marks every missing cell as `True`.
- `.any(axis=1)` asks whether **any cell across each row** is `True`. In pandas, `axis=1` means operate horizontally across columns for each row; `axis=0` means operate vertically down rows for each column.
- `print(...)` sends a value to the output area. When the last line of a Jupyter cell contains only a DataFrame name, Jupyter also renders that DataFrame automatically.

# Part 4 — Build an audit table

An audit table summarizes each column before decisions are made. It should not replace biological reasoning, but it helps us work systematically.

In [ ]:
# Calculate each summary separately.
column_types = raw_data.dtypes
missing_counts = raw_data.isna().sum()
unique_counts = raw_data.nunique()

# Put the three summaries next to one another.
audit_table = pd.DataFrame({
    "data_type": column_types,
    "missing_values": missing_counts,
    "unique_values": unique_counts
})

audit_table

## How the audit code works

Three summaries are calculated separately and then combined into one table.

The table records:

- pandas data type;
- number of missing entries;
- number of distinct non-missing values.

The braces `{...}` create a Python **dictionary**. A dictionary connects each key, such as `"data_type"`, to a value, such as `column_types`. When pandas constructs a DataFrame from this dictionary, the dictionary keys become **column names** and the aligned Series become the columns. The table describes structure; it does not decide whether values are biologically correct.

# Part 5 — Duplicate rows and duplicate identifiers

Two different questions must be separated:

1. Are entire rows exact duplicates?
2. Does an identifier occur more than once?

Repeated identifiers might represent accidental duplication, technical replicates, repeated measures or data-entry mistakes.

In [ ]:
# Check entire rows.
exact_duplicate_rows = raw_data.duplicated()
print("Exact duplicate rows:", exact_duplicate_rows.sum())

# Check the sample_id column separately.
repeated_id = raw_data["sample_id"].duplicated(keep=False)
print("Rows with repeated sample IDs:", repeated_id.sum())

# Display the rows whose IDs are repeated.
rows_with_repeated_id = raw_data[repeated_id]
rows_with_repeated_id

The duplicated `sample_id` requires investigation. The two rows differ in treatment, so blindly dropping one row could discard a genuine sample or conceal a labelling error.

`duplicated()` produces a Boolean Series. By default, the first occurrence is not marked as duplicated. `keep=False` changes that behaviour and marks **every** occurrence of a repeated identifier as `True`. The mask `repeated_id` is then placed inside square brackets to keep those rows. `sum()` counts how many `True` values the mask contains.

<div class="decision-box"><b>Decision principle:</b> flag the records and request clarification from the data source. Do not invent a corrected identifier without evidence.</div>

# Part 6 — Categorical consistency

pandas treats `control`, `Control` and `CONTROL` as different categories. These differences may be biological, or they may be inconsistent spelling.

Always inspect the unique labels before standardizing them.

In [ ]:
print("Species labels:")
print(raw_data["species"].value_counts(dropna=False))

print("\nTreatment labels:")
print(raw_data["treatment"].value_counts(dropna=False))

print("\nBatch labels:")
print(raw_data["batch"].value_counts(dropna=False))

print("\nResponse labels:")
print(raw_data["response"].value_counts(dropna=False))

`value_counts()` counts the labels in one Series. The argument `dropna=False` is important during auditing: it tells pandas to include missing values in the count. Without it, pandas normally omits missing values, which could make the dataset appear more complete than it is.

In this synthetic dataset, `Control` and `control` mean the same treatment, and `Drug` and `drug` mean the same treatment. Lowercase standardization is therefore justified.

A copy is made so the imported data remain unchanged.

In [ ]:
clean_data = raw_data.copy()

# First remove spaces around each treatment label.
treatment_without_spaces = clean_data["treatment"].str.strip()

# Next convert the labels to lowercase.
treatment_lowercase = treatment_without_spaces.str.lower()

# Store the standardized labels in the copied DataFrame.
clean_data["treatment"] = treatment_lowercase

print(clean_data["treatment"].value_counts())

## Why `.str` is necessary

```python
treatment_without_spaces = clean_data["treatment"].str.strip()
```

`clean_data["treatment"]` is a pandas Series containing many treatment labels, not one ordinary Python string. Python's ordinary string methods operate on one string at a time. pandas therefore provides the special `.str` **string accessor**. It gives access to string operations that pandas applies separately to every non-missing value in the Series.

Notice that the syntax is `.str`, not `.str()`. The accessor is not called as a method. After the string accessor, `.strip()` is called.

Read the line from left to right:

1. select the `treatment` column;
2. use `.str` to make pandas string methods available for that Series;
3. call `.strip()` to remove spaces at the beginning and end of every label;
4. store the resulting Series in `treatment_without_spaces`.

The next line:

```python
treatment_lowercase = treatment_without_spaces.str.lower()
```

again uses `.str`, because `treatment_without_spaces` is still an entire Series. `.lower()` converts the letters in every label to lowercase and returns another Series. It does not modify `treatment_without_spaces` itself.

Finally:

```python
clean_data["treatment"] = treatment_lowercase
```

places the standardized Series into the `treatment` column. The same number of values is aligned with the DataFrame rows by their pandas index.

For one ordinary Python string, `.str` would not be used:

```python
one_label = " Drug "
one_clean_label = one_label.strip().lower()
```

Standardization is justified only when the labels are known to have the same meaning.

# Part 7 — Correct a wrongly stored numerical column

`gene_a` became an object column because it contains `not_recorded`. Valid entries will be converted to numbers, while unconvertible text will become missing values.

In [ ]:
# Keep the original text in a new column.
clean_data["gene_a_original"] = clean_data["gene_a"]

# Convert gene_a to numbers.
converted_gene_a = pd.to_numeric(
    clean_data["gene_a"],
    errors="coerce"
)

# Store the converted values.
clean_data["gene_a"] = converted_gene_a

print(clean_data[["sample_id", "gene_a_original", "gene_a"]].tail(10))
print("\nNew dtype:", clean_data["gene_a"].dtype)

The original column is preserved so the transformation can be audited. `errors="coerce"` is powerful but potentially dangerous: spelling mistakes and unit strings can silently become missing. Always count and inspect newly created missing values.

In [ ]:
# Which original entries contained something?
original_was_present = clean_data["gene_a_original"].notna()

# Which converted entries are now missing?
converted_is_missing = clean_data["gene_a"].isna()

# Both must be True for a failed conversion.
conversion_failed = original_was_present & converted_is_missing

failed_rows = clean_data[conversion_failed]
failed_rows[["sample_id", "gene_a_original", "gene_a"]]

`.notna()` asks which values are present, while `.isna()` asks which are missing. The ampersand `&` means **AND** for pandas Boolean Series. Therefore, `conversion_failed` is `True` only where the original value was present **and** the converted value is missing. Parentheses should be placed around each comparison when several comparisons are combined with `&`.

# Part 8 — Explore numerical distributions

A **distribution** describes how values are spread: their centre, range, shape and unusual observations.

Start with numerical summaries.

In [ ]:
numeric_columns = [
    "time_h", "gene_a", "gene_b", "gene_c", "growth_rate"
]

clean_data[numeric_columns].describe().T

## How to read `describe()`

For each numerical column:

- `count` is the number of non-missing values;
- `mean` is the arithmetic average;
- `std` describes spread around the mean;
- `min` and `max` show extremes;
- `25%`, `50%`, `75%` are quartiles;
- `50%` is the median.

`clean_data[numeric_columns]` selects the columns named in the list. `.describe()` calculates the summary table. `.T` is the transpose attribute: it swaps rows and columns so that each biological variable is displayed as one row. Like `.shape`, `.T` is an attribute and therefore has no parentheses.

Large differences between the maximum and upper quartile can signal a possible outlier. A negative minimum may be impossible for a variable such as growth rate, depending on its definition.

## Histograms

A histogram divides a numerical range into intervals called bins and counts observations in each interval. It helps us see skewness, multiple groups and extreme values.

In [ ]:
clean_data[numeric_columns].hist(
    figsize=(12, 8),
    bins=10,
    edgecolor="black"
)
plt.suptitle("Numerical feature distributions", y=1.02)
plt.tight_layout()
plt.show()

### Reading the histogram code

`.hist(...)` asks pandas to draw one histogram for every selected numerical column. `figsize=(12, 8)` supplies a **tuple** containing figure width and height in inches. `bins=10` divides each numerical range into ten intervals. `edgecolor="black"` outlines the bars. The commas separate arguments.

`plt` is the short name for Matplotlib's plotting interface. `plt.suptitle(...)` adds a title above the group of plots, `plt.tight_layout()` reduces overlap, and `plt.show()` displays the completed figure. The option `y=1.02` moves the title slightly upward.

### Interpreting the histograms

Do not report “normal” or “non-normal” from a tiny dataset with certainty. Instead, describe visible patterns:

- `gene_b` has one value far from the others;
- `growth_rate` includes a negative value;
- time is recorded at a few planned time points rather than continuously;
- gene distributions may reflect species and treatment mixtures.

The plots identify questions. They do not automatically answer whether a value is wrong.

## Boxplots

A boxplot displays the median, middle 50% of observations and points beyond conventional whiskers. Points beyond whiskers are **potential outliers**, not automatic errors.

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=clean_data[["gene_a", "gene_b", "gene_c"]])
plt.title("Gene-expression distributions")
plt.ylabel("Normalized expression")
plt.show()

The extreme `gene_b` value compresses the other observations. Possible explanations include:

- a true strong biological response;
- a different measurement unit;
- a decimal-place error;
- sample contamination;
- instrument saturation; or
- data-entry error.

Deleting it without investigation could remove an important biological signal.

### Reading a Seaborn plotting command

`sns.boxplot(...)` calls Seaborn's boxplot function. `data=` identifies the DataFrame to use. When several numeric columns are supplied directly, Seaborn draws one box for each column. In later plots, `x="treatment"` means use that column on the horizontal axis and `y="growth_rate"` means use that column on the vertical axis. The words after `=` are named arguments, not new variables.

## Guided check 2

Create a boxplot of `growth_rate` grouped by treatment. Add a useful title and axis labels. Then write two observations without making a causal claim.

In [ ]:
# Create the boxplot here.

### Guided check 2 — solution

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(data=clean_data, x="treatment", y="growth_rate")
plt.title("Growth rate by treatment")
plt.xlabel("Treatment")
plt.ylabel("Growth rate per hour")
plt.show()

`plt.figure(figsize=(7, 4))` begins a fresh figure. Starting a new figure prevents a new plot from being drawn on top of the previous one. `plt.title`, `plt.xlabel` and `plt.ylabel` add human-readable labels. `plt.show()` finishes and displays the figure.

A defensible observation might be: “In this small synthetic dataset, drug-treated samples generally have lower recorded growth rates than control samples.”

It would be too strong to say that the plot proves the drug causes the difference. Species, batch, time and the deliberate data-quality problems have not yet been accounted for.

# Part 9 — Explore categorical distributions

A count plot shows how many samples belong to each category.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=clean_data, x="species")
plt.title("Samples by species")
plt.xticks(rotation=25)
plt.show()

plt.figure(figsize=(6, 4))
sns.countplot(data=clean_data, x="treatment")
plt.title("Samples by treatment")
plt.show()

plt.figure(figsize=(6, 4))
sns.countplot(data=clean_data, x="response")
plt.title("Samples by response")
plt.show()

Each count plot follows the same four-step pattern: create a new figure, call the plotting function, add a title, and display it. In `sns.countplot(data=clean_data, x="species")`, Seaborn reads the `species` column from `clean_data` and counts its categories. `plt.xticks(rotation=25)` rotates horizontal-axis labels by 25 degrees so long labels do not overlap.

Counts matter because a highly imbalanced category can affect summaries and later model evaluation. Equal total counts also do not guarantee good design: one class might occur mostly in one batch or species.

## Cross-tabulate metadata

`pd.crosstab()` counts combinations of categories.

In [ ]:
print("Treatment by response:")
print(pd.crosstab(clean_data["treatment"], clean_data["response"]))

print("\nBatch by response:")
print(pd.crosstab(clean_data["batch"], clean_data["response"]))

print("\nSpecies by response:")
print(pd.crosstab(clean_data["species"], clean_data["response"]))

`pd.crosstab(first_series, second_series)` constructs a frequency table. Values of the first Series become row labels, values of the second become column labels, and each cell contains the number of observations in that combination. The order of the two Series affects the layout, not the underlying counts.

## Biological interpretation and potential confounding

If treatment and response are perfectly or almost perfectly aligned, predicting `response` using `treatment` may be trivial. The resulting accuracy would not necessarily show that gene-expression features learned a useful biological pattern.

Batch should also be checked. If most resistant samples came from one batch, a model could learn technical differences rather than biology.

Leakage is studied formally in Lecture 3. For now, remember:

> A predictive-looking column may encode the answer or experimental design rather than useful biology.

# Part 10 — Relationships between numerical variables

A scatter plot shows pairs of numerical measurements. Colour can identify categories.

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=clean_data,
    x="gene_a",
    y="growth_rate",
    hue="treatment",
    style="species",
    s=80
)
plt.title("Gene A expression and growth rate")
plt.show()

In `sns.scatterplot`, `x` and `y` select the two numerical axes. `hue="treatment"` assigns colours according to treatment. `style="species"` assigns marker shapes according to species. `s=80` controls marker size; `s` is the parameter name chosen by the plotting library.

Interpret cautiously:

- a visible association does not prove causation;
- groups may separate because treatment affects both variables;
- missing values reduce the plotted sample count;
- species or batch may explain part of the pattern;
- one unusual observation can strongly influence an apparent trend.

## Correlation matrix

Correlation summarizes linear association between numerical variables on a scale from −1 to +1. It does not establish causation and can be distorted by groups and outliers.

In [ ]:
correlation_matrix = clean_data[numeric_columns].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    fmt=".2f"
)
plt.title("Correlation among numerical variables")
plt.show()

Ask before interpreting a correlation:

1. Is the relationship roughly linear?
2. Is one outlier dominating it?
3. Are categories mixed together?
4. Is one variable derived from another?
5. Is the sample size adequate?

A heatmap is a compact exploratory summary, not proof of a mechanism.

### Reading the correlation and heatmap code

`.corr()` calculates pairwise Pearson correlations among the selected numerical columns and returns another DataFrame. In `sns.heatmap`, `annot=True` writes correlation numbers inside the cells; `cmap="coolwarm"` selects a colour palette; `vmin=-1` and `vmax=1` fix the colour scale to the full correlation range; and `fmt=".2f"` displays two digits after the decimal point.

# Part 11 — Missingness

Missingness should be counted by both column and row.

In [ ]:
missing_cells = clean_data.isna()
missing_by_column = missing_cells.sum()
missing_fraction = missing_cells.mean()
missing_percentage = missing_fraction * 100

missing_report = pd.DataFrame({
    "missing_n": missing_by_column,
    "missing_percent": missing_percentage
})

missing_report

`isna()` produces `True` for missing entries. For a Boolean column, pandas treats `True` like 1 and `False` like 0 when calculating a mean. Therefore, `isna().mean()` gives the missing proportion.

Each calculation is stored separately. `missing_cells` is a Boolean DataFrame. Calling `.sum()` on it counts missing cells in each column. Calling `.mean()` calculates the fraction missing in each column, and multiplying by `100` converts a fraction such as `0.10` into `10` percent. The dictionary passed to `pd.DataFrame` gives the two resulting Series clear column names.

In [ ]:
missing_row_mask = clean_data.isna().any(axis=1)
rows_with_missing = clean_data[missing_row_mask]

rows_with_missing

`.any(axis=1)` reduces every row to one Boolean answer: “Does this row contain at least one `True`?” That Boolean Series becomes `missing_row_mask`. Using `clean_data[missing_row_mask]` keeps only rows for which the answer is `True`.

## Visualize the missingness pattern

The heatmap below shows missing cells, not measurement magnitude.

In [ ]:
plt.figure(figsize=(10, 4))
sns.heatmap(
    clean_data.isna(),
    cbar=False,
    yticklabels=False,
    cmap=["white", "darkred"]
)
plt.title("Missing-value pattern: red indicates missing")
plt.xlabel("Column")
plt.show()

For this heatmap, the data are the Boolean table produced by `clean_data.isna()`. `cbar=False` hides the colour bar, `yticklabels=False` hides crowded row labels, and `cmap=["white", "darkred"]` supplies a two-colour list: white for `False` and dark red for `True`.

## Missing-data options

| Option | Possible advantage | Important risk |
|---|---|---|
| Investigate/recover | preserves real information | may not be possible |
| Remove rows | simple | reduces sample size and may bias classes/groups |
| Remove columns | useful if mostly missing | loses an entire measurement |
| Mean imputation | simple | affected by outliers; reduces variation |
| Median imputation | robust to extremes | still inserts estimated values |
| Group-wise imputation | respects known groups | unstable for small groups; may encode group information |
| Missing indicator | records missingness | missingness itself may reflect batch or outcome |

No strategy is automatically correct.

## Demonstration: preserve and impute

New columns are created instead of overwriting the measured values. Missing values are replaced with the median calculated from the available values for each gene.

This is not yet a model-ready pipeline. In a predictive workflow, imputation values must be learned from training data only; this is implemented in Lecture 3.

In [ ]:
# Gene A
clean_data["gene_a_missing"] = clean_data["gene_a"].isna()
gene_a_median = clean_data["gene_a"].median()
clean_data["gene_a_imputed"] = clean_data["gene_a"].fillna(gene_a_median)

# Gene B
clean_data["gene_b_missing"] = clean_data["gene_b"].isna()
gene_b_median = clean_data["gene_b"].median()
clean_data["gene_b_imputed"] = clean_data["gene_b"].fillna(gene_b_median)

# Gene C
clean_data["gene_c_missing"] = clean_data["gene_c"].isna()
gene_c_median = clean_data["gene_c"].median()
clean_data["gene_c_imputed"] = clean_data["gene_c"].fillna(gene_c_median)

print("Gene A median:", gene_a_median)
print("Gene B median:", gene_b_median)
print("Gene C median:", gene_c_median)

### Reading the imputation code

For each gene, the three lines perform three different jobs:

1. `.isna()` creates a permanent Boolean indicator showing which original measurements were missing.
2. `.median()` calculates the middle available value. pandas ignores missing values for this calculation.
3. `.fillna(gene_a_median)` returns a new Series in which missing values are replaced by the supplied median.

The result is assigned to a **new** column ending in `_imputed`; the measured column is not overwritten. `fillna()` does not change the original Series here because its returned result is stored elsewhere.

The original gene columns still contain missing values. The new `_imputed` columns contain substituted values, and the `_missing` columns record which entries were originally absent. This preserves an audit trail.

In [ ]:
gene_columns = ["gene_a", "gene_b", "gene_c"]
missing_gene_mask = clean_data[gene_columns].isna().any(axis=1)
rows_with_missing_genes = clean_data[missing_gene_mask]

columns_to_show = [
    "sample_id",
    "gene_a", "gene_a_imputed", "gene_a_missing",
    "gene_b", "gene_b_imputed", "gene_b_missing",
    "gene_c", "gene_c_imputed", "gene_c_missing"
]

rows_with_missing_genes[columns_to_show]

`gene_columns` and `columns_to_show` are ordinary Python lists. Supplying `gene_columns` inside `clean_data[...]` selects those three columns. After the rows are filtered, supplying `columns_to_show` selects and orders the fields displayed in the final table.

# Part 12 — Implausible values and outlier flags

An **implausible value** violates known measurement rules. A **statistical outlier** is unusual compared with the observed distribution. These are not the same.

If growth rate is defined so that negative values are impossible in this experiment, a negative value should be flagged as implausible.

In [ ]:
clean_data["growth_rate_implausible"] = clean_data["growth_rate"] < 0

implausible_rows = clean_data[clean_data["growth_rate_implausible"]]
columns_to_show = ["sample_id", "growth_rate", "species", "treatment", "batch"]
implausible_rows[columns_to_show]

The comparison `< 0` is performed separately for every value in the `growth_rate` Series. Its result is a Boolean Series stored as a new quality-control column. This is **flagging**, not deletion: the original value remains available for investigation.

The program flags the row but does not silently remove or replace it. The next action should be guided by laboratory records and measurement definitions.

## IQR rule for potential outliers

The interquartile range is:

$$IQR = Q_3 - Q_1$$

A common screening rule flags values below $Q_1 - 1.5\times IQR$ or above $Q_3 + 1.5\times IQR$.

This is a statistical convention, not a biological law.

In [ ]:
gene_b_q1 = clean_data["gene_b"].quantile(0.25)
gene_b_q3 = clean_data["gene_b"].quantile(0.75)
gene_b_iqr = gene_b_q3 - gene_b_q1

gene_b_lower = gene_b_q1 - (1.5 * gene_b_iqr)
gene_b_upper = gene_b_q3 + (1.5 * gene_b_iqr)

below_lower_limit = clean_data["gene_b"] < gene_b_lower
above_upper_limit = clean_data["gene_b"] > gene_b_upper
clean_data["gene_b_outlier"] = below_lower_limit | above_upper_limit

print("Lower IQR limit:", gene_b_lower)
print("Upper IQR limit:", gene_b_upper)

gene_b_outlier_rows = clean_data[clean_data["gene_b_outlier"]]
columns_to_show = ["sample_id", "gene_b", "species", "treatment", "batch"]
gene_b_outlier_rows[columns_to_show]

### Reading the IQR code

- `.quantile(0.25)` calculates the first quartile, Q1; `.quantile(0.75)` calculates Q3.
- Subtraction and multiplication follow normal arithmetic rules. Parentheses in `(1.5 * gene_b_iqr)` make the intended grouping obvious.
- `<` and `>` create two Boolean Series.
- The vertical bar `|` means element-by-element **OR** for pandas Boolean Series. A value is flagged when it is below the lower limit **or** above the upper limit.
- The Boolean flag is stored in `clean_data["gene_b_outlier"]` so it remains connected to the appropriate row.

## Outlier decision record

For the extreme `gene_b` value, a defensible report should state:

1. how it was detected;
2. its sample ID and metadata;
3. whether the unit and raw record were checked;
4. whether it is retained, corrected or excluded;
5. how the analysis changes with and without it.

Because this synthetic dataset supplies no laboratory record, the original value is retained and flagged without claiming that it is erroneous.

## Guided check 3

Repeat the IQR steps for `growth_rate`. Calculate Q1, Q3, the IQR and the two limits. Then create a Boolean column called `growth_rate_outlier`. Each calculation should be stored separately so that its value can be examined.

In [ ]:
# Write your code here.

### Guided check 3 — solution

In [ ]:
growth_q1 = clean_data["growth_rate"].quantile(0.25)
growth_q3 = clean_data["growth_rate"].quantile(0.75)
growth_iqr = growth_q3 - growth_q1

growth_lower = growth_q1 - (1.5 * growth_iqr)
growth_upper = growth_q3 + (1.5 * growth_iqr)

growth_below_limit = clean_data["growth_rate"] < growth_lower
growth_above_limit = clean_data["growth_rate"] > growth_upper
clean_data["growth_rate_outlier"] = growth_below_limit | growth_above_limit

print("Lower limit:", growth_lower)
print("Upper limit:", growth_upper)

growth_outlier_rows = clean_data[clean_data["growth_rate_outlier"]]
growth_outlier_rows[["sample_id", "growth_rate"]]

# Part 13 — Biological metadata and experimental structure

Metadata should not be treated as decoration. It describes how samples were generated.

Important questions include:

- Were samples measured in different batches?
- Are multiple rows related to the same biological source?
- Are time points repeated measurements from the same organism or culture?
- Is treatment balanced within species and batch?
- Was one class measured mostly on one day or instrument?

Ignoring structure can produce excellent-looking but misleading results.

In [ ]:
batch_treatment_table = pd.crosstab(
    clean_data["batch"],
    clean_data["treatment"]
)

batch_treatment_table

`pd.crosstab(clean_data["batch"], clean_data["treatment"])` counts every batch–treatment combination. The first Series supplies table rows and the second supplies table columns. The returned DataFrame is stored in `batch_treatment_table` before it is displayed.

A table with zero counts in some combinations signals imbalance. Later splitting and validation must respect related samples, batches and repeated measures. At the exploration stage, the structure should be detected and reported.

# Part 14 — Encoding categorical variables

Most scikit-learn estimators require numerical input. Category labels such as `E_coli` cannot simply be passed as text.

**One-hot encoding** creates a binary indicator column for each category.

In [ ]:
categorical_features = ["species", "batch"]

encoded_categories = pd.get_dummies(
    clean_data[categorical_features],
    columns=categorical_features,
    dtype=int
)

encoded_categories.head()

### Reading the encoding code

`categorical_features` is a list containing the columns selected for encoding. `clean_data[categorical_features]` selects a small DataFrame containing only those columns. `pd.get_dummies(...)` converts categories into indicator columns. The argument `columns=categorical_features` explicitly names the columns to encode, while `dtype=int` requests integer `0` and `1` values instead of Boolean `False` and `True`. The encoded result is stored separately so the original category labels remain available.

## How to interpret one-hot columns

For one row:

- `species_E_coli = 1` means the sample belongs to *E. coli*;
- other species indicator columns are 0;
- batch indicators similarly record the batch.

One-hot encoding does not claim that one species is numerically larger than another. Assigning `E_coli=1`, `B_subtilis=2`, `P_aeruginosa=3` would create a false numerical order.

## Should treatment be encoded?

That depends on the scientific question. If `response` is the future target and treatment nearly determines it, including treatment could make prediction trivial or scientifically unhelpful. Feature decisions are examined explicitly in Lecture 3.

Here, encoding is demonstrated only for `species` and `batch`, while identifiers and targets remain separate.

# Part 15 — Scaling numerical features

Features can have different units and ranges. For example, time may range from 0 to 48 while normalized expression values are around 2–10.

Some algorithms use distances or coefficient penalties and can be dominated by features with larger numerical scales.

Standard scaling transforms a value $x$ using:

$$z = \frac{x - \mu}{\sigma}$$

where $\mu$ is the feature mean and $\sigma$ is its standard deviation.

## What standard scaling does and does not do

It generally makes each scaled feature have approximately:

- mean 0;
- standard deviation 1.

It does **not**:

- make a skewed distribution normal;
- remove outliers;
- make a measurement biologically valid;
- select useful features;
- prevent leakage automatically.

Outliers can strongly affect the mean and standard deviation.

In [ ]:
numeric_features_for_demo = [
    "time_h",
    "gene_a_imputed",
    "gene_b_imputed",
    "gene_c_imputed",
    "growth_rate"
]

scaler = StandardScaler()

scaled_array = scaler.fit_transform(
    clean_data[numeric_features_for_demo]
)

scaled_numeric = pd.DataFrame(
    scaled_array,
    columns=numeric_features_for_demo,
    index=clean_data.index
)

scaled_numeric.head()

## How to read the scaling code

1. `StandardScaler()` creates a scaler object.
2. `.fit_transform()` calculates means and standard deviations, then transforms the data.
3. scikit-learn returns a NumPy array without column labels.
4. `pd.DataFrame(...)` restores meaningful labels and the original row index.

In the DataFrame constructor, `columns=numeric_features_for_demo` restores the feature names. `index=clean_data.index` restores the original row labels so that scaled rows remain aligned with their samples. A NumPy array has values but does not carry pandas column or index labels.

<div class="warning-box"><b>Leakage warning:</b> this full-dataset scaling is only a visualization and mechanics demonstration. In a predictive workflow, the scaler must be fitted on training data and then applied to validation/test data. Lecture 3 will implement that correctly with a pipeline.</div>

In [ ]:
print("Scaled means:")
print(scaled_numeric.mean().round(6))

print("\nScaled standard deviations (population definition):")
print(scaled_numeric.std(ddof=0).round(6))

`.mean()` calculates a mean for each scaled column. `.std(ddof=0)` calculates the population standard deviation, matching the definition used by `StandardScaler`; `ddof` means “delta degrees of freedom.” `.round(6)` rounds the displayed results to six decimal places. It changes presentation here, not the values stored in `scaled_numeric`.

The means are approximately zero and standard deviations approximately one. Tiny values such as `−0.000000` are floating-point rounding effects.

# Part 16 — Construct a demonstration preprocessing table

The preprocessing report combines:

- identifiers and metadata for traceability;
- original cleaned categories;
- imputed numerical features;
- missingness indicators;
- quality-control flags;
- one-hot category indicators.

The raw imported table is not overwritten.

In [ ]:
report_columns = [
    "sample_id", "species", "treatment", "batch", "response",
    "time_h", "growth_rate",
    "gene_a_imputed", "gene_b_imputed", "gene_c_imputed",
    "gene_a_missing", "gene_b_missing", "gene_c_missing",
    "growth_rate_implausible", "gene_b_outlier"
]

preprocessing_table = clean_data[report_columns].copy()
preprocessing_table = preprocessing_table.join(encoded_categories)

preprocessing_table.head()

`clean_data[report_columns]` selects and orders the report fields. `.copy()` creates an independent DataFrame. `.join(encoded_categories)` adds the encoded columns by matching their row indexes. The joined result is assigned back to `preprocessing_table`; without this assignment, the returned joined table would not be retained.

## Why retain identifiers if they are not model features?

Identifiers are needed to trace results back to samples, investigate problems and join tables. They should be retained in the report but excluded from the feature matrix used by most models.

Keeping a column in a preprocessing report does not mean feeding it to a model.

# Part 17 — A preprocessing decision log

A reproducible analysis records what was found and what was done.

In [ ]:
issues = [
    "Inconsistent treatment capitalization",
    "gene_a stored as object",
    "Missing gene measurements",
    "Duplicated sample ID",
    "Extreme gene_b value",
    "Negative growth rate"
]

decisions = [
    "Strip spaces and convert to lowercase",
    "Keep the original and convert invalid text to missing",
    "Create median-imputed columns and missing indicators",
    "Flag it and request clarification",
    "Retain the original value and flag it",
    "Flag it and verify the definition and source"
]

reasons = [
    "The labels have the same documented meaning",
    "The failed conversion remains visible",
    "Rows are preserved for this preprocessing example",
    "It may be an error or two distinct samples",
    "There is no laboratory evidence that it is wrong",
    "Values should not be silently corrected or removed"
]

decision_log = pd.DataFrame({
    "issue": issues,
    "decision": decisions,
    "reason": reasons
})

decision_log

The three Python lists have the same length and matching positions: the first issue belongs with the first decision and first reason. The dictionary passed to `pd.DataFrame` turns those lists into three named columns. If the lists had different lengths, pandas would raise an error because it could not form rectangular rows.

A decision log should be specific enough that another researcher can understand and challenge each choice. “Cleaned the data” is not adequate documentation.

# Part 18 — Save outputs reproducibly

Save the preprocessing report and decision log without writing the DataFrame index as an extra column.

In [ ]:
preprocessing_table.to_csv(
    "lecture02_preprocessing_report.csv",
    index=False
)

decision_log.to_csv(
    "lecture02_preprocessing_decisions.csv",
    index=False
)

print("Saved preprocessing report and decision log.")

`.to_csv(...)` writes a DataFrame to a comma-separated text file. The first argument is the output filename. `index=False` prevents pandas from writing its internal row index as an extra CSV column. The DataFrame in memory is not removed after saving.

## Verify saved files

Writing a file is not the final check. Reopen it and compare basic properties.

In [ ]:
reloaded_report = pd.read_csv("lecture02_preprocessing_report.csv")

print("Original report shape:", preprocessing_table.shape)
print("Reloaded report shape:", reloaded_report.shape)
print("Columns identical:", reloaded_report.columns.tolist() == preprocessing_table.columns.tolist())

The saved report is read into a new variable rather than assuming that writing succeeded. The operator `==` asks whether the two column-name lists are equal in both content and order. Unlike assignment with one equals sign, comparison with two equals signs returns `True` or `False`.

# Part 19 — Guided practical

Complete the following tasks before viewing the suggested solutions.

## Task A: structural audit

1. Report rows and columns.
2. Identify numerical and non-numerical columns.
3. Count missing values.
4. Identify repeated sample IDs.
5. List unique labels in all categorical columns.

## Task B: visual exploration

1. Plot each gene distribution.
2. Compare each gene by treatment using boxplots.
3. Plot growth rate against `gene_b`, coloured by species.
4. Create a count plot for batch.
5. Write one observation and one limitation for each plot.

## Task C: quality flags

1. Flag missing gene measurements.
2. Flag negative growth rates.
3. Apply the IQR screen to all genes.
4. Create one table containing all flagged rows.
5. Explain why flagging is not the same as deleting.

## Task D: preprocessing reasoning

1. Identify columns that should not become model features.
2. Identify categorical columns that may require encoding.
3. Identify numerical columns that may require scaling.
4. Explain when imputation and scaling must be fitted in a future ML workflow.
5. Write three additions to the decision log.

In [ ]:
# Complete the guided practical here or in additional cells.

# Part 20 — Suggested guided-practical code

Try the tasks first. The code below is one defensible approach, not the only possible analysis.

In [ ]:
# Task A
print(clean_data.shape)
print(clean_data.select_dtypes(include="number").columns.tolist())
print(clean_data.select_dtypes(exclude="number").columns.tolist())
print(clean_data.isna().sum())
repeated_id_mask = clean_data["sample_id"].duplicated(keep=False)
print(clean_data[repeated_id_mask])

print("Species:", clean_data["species"].unique())
print("Treatments:", clean_data["treatment"].unique())
print("Batches:", clean_data["batch"].unique())
print("Responses:", clean_data["response"].unique())

`select_dtypes(include="number")` keeps columns pandas recognizes as numerical. `exclude="number"` keeps the remaining columns. The result is a DataFrame; `.columns` obtains its column names, and `.tolist()` converts those names to an ordinary list. This is useful for an audit, but the stored data type alone does not determine a column's scientific role.

In [ ]:
# Task B: gene A boxplot
plt.figure(figsize=(7, 4))
sns.boxplot(data=clean_data, x="treatment", y="gene_a")
plt.title("Gene A by treatment")
plt.show()

# Task B: scatter plot
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=clean_data,
    x="gene_b",
    y="growth_rate",
    hue="species",
    s=80
)
plt.title("Growth rate and gene B expression")
plt.show()

These plots use the same sequence introduced earlier: `create figure → choose plot and data → label → show`. Recognizing this repeated pattern makes it easier to read other Matplotlib and Seaborn plots.

In [ ]:
# Task C: create one clear flag at a time
missing_gene = clean_data[["gene_a", "gene_b", "gene_c"]].isna().any(axis=1)
implausible_growth = clean_data["growth_rate_implausible"]
unusual_gene_b = clean_data["gene_b_outlier"]
repeated_sample_id = clean_data["sample_id"].duplicated(keep=False)

# The | symbol means OR.
combined_flag = missing_gene | implausible_growth | unusual_gene_b | repeated_sample_id

flagged_rows = clean_data[combined_flag].copy()
flagged_rows

Task C gives each Boolean condition a descriptive name. The vertical bar `|` combines the conditions row by row and means OR: a row is selected if at least one condition is `True`. The expression is long but contains no hidden loop. `.copy()` makes the resulting flagged-row table independent of the source DataFrame.

# Part 21 — AI-assisted preprocessing audit

An AI assistant proposes:

```python
df = pd.read_csv("lecture02_bacterial_expression_preprocessing.csv")
df = df.dropna()
df = df.drop_duplicates()
df["species"] = df["species"].map({
    "E_coli": 1,
    "B_subtilis": 2,
    "P_aeruginosa": 3
})
df = (df - df.mean()) / df.std()
print("Data ready for machine learning")
```

Identify at least ten problems, uncertainties or unsupported decisions.

## Audit prompts

Consider:

1. Does `dropna()` investigate or report removed samples?
2. Does `drop_duplicates()` detect duplicated identifiers with different measurements?
3. Does integer encoding create a false species order?
4. Can subtraction be applied to text columns?
5. Are identifiers, metadata and target separated from features?
6. Are implausible values investigated?
7. Are outliers handled or documented?
8. Is scaling fitted before a train/test split?
9. Are biological batches and related samples considered?
10. What evidence supports “ready for machine learning”?

### Your AI-code audit

Write the problems, consequences and corrections here.

# Part 22 — Independent formative check

Complete without AI or another person.

1. Load the original CSV into a new DataFrame.
2. Report its shape and types.
3. Standardize treatment capitalization without changing the original DataFrame.
4. Convert `gene_a` safely and report failed conversions.
5. Calculate missing counts and percentages.
6. Identify duplicated sample IDs.
7. Make one histogram, one boxplot, one count plot and one scatter plot.
8. Flag the negative growth rate.
9. Explain why the extreme `gene_b` value should not be deleted automatically.
10. Explain why full-dataset scaling must not be used before evaluation in a future predictive workflow.

In [ ]:
# Complete the independent check here.

# Part 23 — Take-home exercise

Create:

```text
SBI812_Lecture02_Exercise_YourName.ipynb
```

Using the supplied CSV, produce a short exploratory and preprocessing report containing:

1. a biological question;
2. a data dictionary assigning roles to all columns;
3. structural inspection;
4. missingness analysis;
5. duplicate-ID analysis;
6. category consistency checks;
7. at least four clearly labelled visualizations;
8. outlier and plausibility flags;
9. a preprocessing decision log;
10. a demonstration of one-hot encoding;
11. a demonstration of scaling with a leakage warning;
12. saved output tables; and
13. a 200–300 word interpretation discussing patterns, limitations and unresolved issues.

The notebook must run from top to bottom and must not claim that preprocessing choices are biologically correct without justification.

# Part 24 — Review questions

1. What is exploratory data analysis?
2. Why should the imported DataFrame be preserved?
3. What is the difference between missing, implausible and statistically unusual?
4. Why can inconsistent capitalization matter?
5. Why is an identifier normally excluded from model features?
6. Why might batch be important?
7. What information does a histogram provide?
8. Why is an outlier flag not an instruction to delete?
9. What does one-hot encoding accomplish?
10. Why is arbitrary integer encoding dangerous for unordered categories?
11. What does standard scaling do?
12. What does scaling not do?
13. Why should preprocessing later be fitted using training data only?
14. What should a preprocessing decision log contain?
15. What evidence would justify saying that a dataset is ready for modelling?

## Review answers

1. A structured investigation of data quality, distributions, relationships and experimental structure before modelling.
2. To preserve the source as loaded and make changes auditable.
3. Missing means absent; implausible conflicts with known rules; unusual is distant from most observations but may be real.
4. pandas treats differently capitalized labels as separate categories.
5. It names a sample rather than measuring biology and may allow memorization.
6. Technical differences may be mistaken for biological patterns.
7. The frequency distribution across numerical intervals.
8. Statistical unusualness does not prove error.
9. It creates numerical indicator columns without imposing category order.
10. Numbers imply an order and distance that may not exist.
11. It centres a feature and expresses values in standard-deviation units.
12. It does not validate, remove outliers, normalize shape or choose useful features.
13. Using full data lets information from evaluation samples influence preprocessing.
14. Issue, evidence, decision, reason and ideally effect on sample/feature counts.
15. Documented types, meanings, quality issues, decisions, unresolved limitations and a feature/target plan appropriate to the biological question.

# Key terms

| Term | Meaning |
|---|---|
| Exploratory data analysis | Inspection of structure, quality, distributions and relationships |
| Distribution | How numerical observations are spread |
| Missingness | Pattern and amount of absent data |
| Imputation | Substitution of an estimated value for a missing entry |
| Outlier | Observation unusually far from most others by a stated rule |
| Implausible value | Value conflicting with known measurement or biological constraints |
| Metadata | Information describing samples or experimental generation |
| Confounder | Factor associated with both a predictor and outcome that can mislead interpretation |
| Encoding | Conversion of categories into a numerical representation |
| One-hot encoding | Separate 0/1 indicator columns for categories |
| Scaling | Transformation of numerical feature magnitude |
| Standardization | Subtract mean and divide by standard deviation |
| Leakage | Illegitimate use of information unavailable in real future prediction |
| Decision log | Record of detected issues, actions and reasons |
| Audit trail | Information allowing transformations to be traced and checked |

# Lecture summary

Exploration and preprocessing are scientific reasoning tasks supported by code. This lecture covered how to:

- loaded and preserved supplied biological data;
- audited structure, types, categories and missingness;
- visualized numerical and categorical distributions;
- identified duplicate identifiers and inconsistent labels;
- distinguished unusual from implausible values;
- preserved original measurements while creating flags and imputed demonstration columns;
- introduced one-hot encoding and standard scaling;
- documented decisions; and
- saved and verified preprocessing reports.

Lecture 3 will use these foundations to construct a proper machine-learning workflow with feature/target separation, train/test splitting, baselines, pipelines, overfitting awareness and leakage prevention.